In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py, os, tqdm, glob, scipy
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.49'
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

import jax
import jax.numpy as jnp
import jax_cosmo as jc

import optax
from optax.losses import huber_loss
from flax import nnx
import orbax.checkpoint as ocp
import jraph

import diffrax
from diffrax import diffeqsolve, ODETerm, LeapfrogMidpoint, PIDController, SaveAt, ConstantStepSize

import jaxpm
from jaxpm.painting import cic_paint, cic_read, compensate_cic
from jaxpm.kernels import fftk, gradient_kernel, invlaplace_kernel, longrange_kernel, invnabla_kernel
from jaxpm.utils import power_spectrum, cross_correlation_coefficients
from jaxpm.nn import MLP, CNN, HybridNet, AttentionGNN, ResidualMLP
from jaxpm import camels, plotting, hpm, nn, graph, diagnostics

print(jax.default_backend())

gpu


# configuration

In [3]:
parts_per_dim = 64
mesh_per_dim = parts_per_dim

# parts_per_dim = 128
# mesh_per_dim = parts_per_dim

# parts_per_dim = None
# mesh_per_dim = 256

mesh_shape = [mesh_per_dim] * 3
box_size = [float(mesh_per_dim)] * 3

# i_snapshots = None
# i_snapshots = range(1, 33+4, 8)
# i_snapshots = np.arange(-8, 0)
i_snapshots = np.arange(-4, 0)

# CAMELS

In [4]:
train_dict = camels.load_CV_snapshots(
    "CV_0",
    # "CV_1",
    mesh_per_dim,
    parts_per_dim,
    i_snapshots=i_snapshots,
    return_hydro=True,
    CODE="Astrid",
    # CODE="IllustrisTNG",
)

cosmo = train_dict["cosmo"]
scales = train_dict["scales"]

dm_poss = train_dict["dm_poss"]
dm_vels = train_dict["dm_vels"]

gas_poss = train_dict["gas_poss"]
gas_vels = train_dict["gas_vels"]

No matching catalogs found, returning only the snapshots
Using snapshots ['/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/Astrid/CV/CV_0/snapshot_084.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/Astrid/CV/CV_0/snapshot_086.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/Astrid/CV/CV_0/snapshot_088.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/Astrid/CV/CV_0/snapshot_090.hdf5']
Selecting 262144 dark matter (deterministic)
Selecting 262144 gas particles (random)


finding unique gas particle indices: 100%|██████████| 4/4 [00:20<00:00,  5.09s/it]


There are 16706476 (99.58%) gas particles that exist in all snapshots


loading snapshots: 100%|██████████| 4/4 [00:30<00:00,  7.69s/it]


In [5]:
vcic_paint = jax.vmap(cic_paint, in_axes=(None,0,None))
vcic_read = jax.vmap(cic_read, in_axes=(0,0))

In [6]:
@nnx.jit(static_argnames=("loss_fn",))
def train_step(model, optimizer, loss_fn):
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)

    return loss

losses = []

In [7]:
edges = graph.get_edges(gas_poss, scales, k=4)

def solve_ode_diffrax(model, architecture, with_latent=True):
    y0 = (dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0])
    
    res = diffeqsolve(
            terms=ODETerm(hpm.get_hpm_network_ode_fn(mesh_per_dim, cosmo, gravity_model=None, pressure_model=model, gas_architecture=architecture, precomputed_edges=edges)),
            solver=LeapfrogMidpoint(),
            t0=scales[0],
            t1=scales[-1],
            # dt0=0.02,
            dt0=0.01,
            y0=y0,
            saveat=SaveAt(ts=scales),
            max_steps=100,
            stepsize_controller=ConstantStepSize(),
        )
    res = res.ys

    return res

# loss

### CAMELS ground truth

In [8]:
# per-particle reference
ref_pos = jnp.stack(gas_poss, axis=0)
ref_vel = jnp.stack(gas_vels, axis=0)

# # velocity
# vel_mean_ref = jax.vmap(
#     jax.vmap(
#         cic_paint, 
#         in_axes=(None,0,0)
#     ),
#     in_axes=(None,None,-1), out_axes=-1,
# )(jnp.zeros(mesh_shape), ref_pos, ref_vel / ref_N[..., jnp.newaxis])

# ref_vel_mean = jax.vmap(
#     jax.vmap(
#         cic_read, 
#         in_axes=(0,0)
#     ),
#     in_axes=(-1,None), out_axes=-1,
# )(vel_mean_ref, ref_pos)

# ref_vel_disp = jnp.sum((ref_vel_mean - ref_vel) ** 2, axis=-1)

# field-level reference
gas_mass = cosmo.Omega_b / (cosmo.Omega_b + cosmo.Omega_c)
ref_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)

# power spectrum reference
vpower_spectrum = jax.vmap(
    lambda fields: 
        power_spectrum(
            compensate_cic(fields),
            boxsize=np.array([25.0] * 3),
            kmin=np.pi / 25.0,
            dk=2 * np.pi / 25.0,
        )
)
_, ref_cls = vpower_spectrum(ref_rho)

vcross_correlation_separate = jax.vmap(
    lambda field_a, field_b:
        cross_correlation_coefficients(
            compensate_cic(field_a),
            compensate_cic(field_b),
            boxsize=np.array([25.0] * 3),
            kmin=np.pi / 25.0,
            dk=2 * np.pi / 25.0,
        )
)
vcross_correlation = lambda rhos: vcross_correlation_separate(rhos, ref_rho)

### particle-level

In [9]:
# def particle_loss_fn(model, architecture, eps=1e-8):
#     res = solve_ode_diffrax(model, architecture)
#     gas_poss = res[2]
#     gas_vels = res[3]

#     delta_pos = ((gas_poss - ref_pos + mesh_per_dim // 2) % mesh_per_dim) - mesh_per_dim // 2
#     pos_loss = jnp.sum(delta_pos**2, axis=-1)

#     # pos_loss /= jnp.maximum(jnp.mean(pos_loss, axis=1, keepdims=True), eps)
#     pos_loss /= jnp.maximum(scales[:, jnp.newaxis]**3, eps)
    
#     pos_loss = jnp.mean(pos_loss)
    
#     res_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)
#     kbins, res_cls = vpower_spectrum(res_rho)

#     k = kbins[0]
#     k_min, k_cutoff = k[0], k[int(0.2*len(k))]
#     k_weights = jnp.expand_dims(jnp.exp(-(k - k_min) / k_cutoff), 0)

#     cl_loss = jnp.sum(((res_cls/jnp.maximum(ref_cls, eps) - 1)*k_weights)**2, axis=-1)

#     # cl_loss /= jnp.mean(cl_loss, axis=0, keepdims=True)
#     # cl_loss /= jnp.maximum(scales[:,jnp.newaxis]**2, eps)
    
#     cl_loss = jnp.mean(cl_loss)

#     # return pos_loss + 0.1 * cl_loss
#     return pos_loss
#     # return cl_loss



In [10]:
# def particle_loss_fn(model, architecture, huber=True, pos_dead_zone=True):
#     res = solve_ode_diffrax(model, architecture)
#     gas_poss = res[2]
#     gas_vels = res[3]

#     delta_pos = ((gas_poss - ref_pos + mesh_per_dim // 2) % mesh_per_dim) - mesh_per_dim // 2
#     if huber:
#         pos_loss = jnp.sum(huber_loss(delta_pos), axis=-1)
#     else:
#         pos_loss = jnp.sum(delta_pos**2, axis=-1)
#     if pos_dead_zone:
#         pos_loss = jnp.where(jnp.sqrt(pos_loss) < 1/mesh_per_dim, 0.0, pos_loss)
#     pos_loss = jnp.mean(pos_loss)
    
#     res_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)
#     kbins, res_cls = vpower_spectrum(res_rho)

#     k = kbins[0]
#     k_min, k_cutoff = k[0], k[int(0.2*len(k))]
#     k_weights = jnp.expand_dims(jnp.exp(-(k - k_min) / k_cutoff), 0)

#     cl_loss = jnp.sum(((res_cls/ref_cls - 1)*k_weights)**2, axis=-1)
#     cl_loss = jnp.mean(cl_loss)

#     return pos_loss + 0.1 * cl_loss


### field-level

In [11]:
def field_loss_fn(model, architecture, eps=1e-8):
    print("using voxel MSE")
    
    res = solve_ode_diffrax(model, architecture)
    gas_poss = res[2]%mesh_per_dim
    
    rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, gas_mass)

    loss = (rho - ref_rho)**2
    # loss /= jnp.maximum(scales.reshape(-1,1,1,1)**2, eps)
    loss = jnp.mean(loss)
        
    return loss

# architecture

### MLP

In [12]:
# model = MLP(
#     d_in=5,
#     d_out=1, 
#     d_hidden=64, 
#     n_hidden=4, 
#     dropout_rate=0.0,
#     rngs=nnx.Rngs(0),
#     norm_type="batch",
# )
# architecture = "mlp"

### CNN

In [13]:
# model = CNN(
#     d_in=5, 
#     d_out=1,
#     d_hidden=64,
#     # n_hidden=8,
#     n_hidden=4,
#     # kernel_size=(5, 5, 5),
#     kernel_size=(3, 3, 3),
#     strides=1,
#     rngs=nnx.Rngs(0),
#     use_residual=False,
#     # use_residual=True,
#     # norm_type=False,
#     # norm_type="batch",
#     # norm_type="layer",
# )

# architecture = "cnn"

In [14]:
# class ResNetBlock(nnx.Module):
#     """Residual block for 3D data with circular padding."""
    
#     def __init__(
#         self,
#         channels: int,
#         kernel_size: tuple = (3, 3, 3),
#         strides: int = 1,
#         rngs: nnx.Rngs = nnx.Rngs(0),
#         activation=jax.nn.relu,
#         norm_type: str = "batch",
#         use_residual: bool = True,
#     ):
#         self.channels = channels
#         self.strides = strides
#         self.activation = activation
#         self.use_residual = use_residual
#         self.norm_type = norm_type
        
#         # Main path convolutions
#         self.conv1 = nnx.Conv(channels, channels, kernel_size, strides, padding="CIRCULAR", rngs=rngs)
#         self.conv2 = nnx.Conv(channels, channels, kernel_size, 1, padding="CIRCULAR", rngs=rngs)
        
#         # Normalization layers
#         if norm_type == "batch":
#             self.norm1 = nnx.BatchNorm(channels, rngs=rngs)
#             self.norm2 = nnx.BatchNorm(channels, rngs=rngs)
#         elif norm_type == "layer":
#             self.norm1 = nnx.LayerNorm(channels, rngs=rngs)
#             self.norm2 = nnx.LayerNorm(channels, rngs=rngs)
        
#         # Skip connection (for when strides > 1)
#         if strides > 1:
#             self.skip_conv = nnx.Conv(channels, channels, (1, 1, 1), strides, padding="CIRCULAR", rngs=rngs)
#             if norm_type == "batch":
#                 self.skip_norm = nnx.BatchNorm(channels, rngs=rngs)
#             elif norm_type == "layer":
#                 self.skip_norm = nnx.LayerNorm(channels, rngs=rngs)
    
#     def __call__(self, x, training: bool = False):
#         # Save input for residual connection
#         residual = x
        
#         # First convolution block
#         y = self.conv1(x)
#         if self.norm_type == "batch":
#             y = self.norm1(y, use_running_average=not training)
#         elif self.norm_type == "layer":
#             y = self.norm1(y)
#         y = self.activation(y)
        
#         # Second convolution block
#         y = self.conv2(y)
#         if self.norm_type == "batch":
#             y = self.norm2(y, use_running_average=not training)
#         elif self.norm_type == "layer":
#             y = self.norm2(y)
        
#         # Apply skip connection if using residual and shapes match
#         if self.use_residual:
#             # If strides > 1, we need to downsample the residual
#             if hasattr(self, 'skip_conv'):
#                 residual = self.skip_conv(residual)
#                 if self.norm_type == "batch":
#                     residual = self.skip_norm(residual, use_running_average=not training)
#                 elif self.norm_type == "layer":
#                     residual = self.skip_norm(residual)
            
#             y = y + residual
        
#         # Final activation after adding the residual
#         return self.activation(y)


# class ResNet(nnx.Module):
#     """3D ResNet implementation for voxel fields with circular padding."""
    
#     def __init__(
#         self,
#         in_channels: int,
#         hidden_channels: int,
#         out_channels: int,
#         n_blocks: int,
#         kernel_size: tuple = (3, 3, 3),
#         strides: int = 1,
#         rngs: nnx.Rngs = nnx.Rngs(0),
#         activation=jax.nn.relu,
#         norm_type: str = "batch",
#         use_residual: bool = True,
#     ):
#         self.in_channels = in_channels
#         self.hidden_channels = hidden_channels
#         self.out_channels = out_channels
#         self.activation = activation
#         self.norm_type = norm_type
        
#         # Input projection
#         self.conv_in = nnx.Conv(in_channels, hidden_channels, kernel_size, 1, padding="CIRCULAR", rngs=rngs)
#         if norm_type == "batch":
#             self.norm_in = nnx.BatchNorm(hidden_channels, rngs=rngs)
#         elif norm_type == "layer":
#             self.norm_in = nnx.LayerNorm(hidden_channels, rngs=rngs)
        
#         # Residual blocks
#         self.blocks = [
#             ResNetBlock(
#                 hidden_channels,
#                 kernel_size=kernel_size,
#                 strides=strides if i == 0 else 1,  # Apply stride only to first block if needed
#                 rngs=rngs,
#                 activation=activation,
#                 norm_type=norm_type,
#                 use_residual=use_residual,
#             )
#             for i in range(n_blocks)
#         ]
        
#         # Output projection
#         self.conv_out = nnx.Conv(hidden_channels, out_channels, kernel_size, 1, padding="CIRCULAR", rngs=rngs)
#         if norm_type == "batch":
#             self.norm_out = nnx.BatchNorm(out_channels, rngs=rngs)
#         elif norm_type == "layer":
#             self.norm_out = nnx.LayerNorm(out_channels, rngs=rngs)
    
#     def __call__(self, x, training: bool = False):
#         # Input projection
#         x = self.conv_in(x)
#         if self.norm_type == "batch":
#             x = self.norm_in(x, use_running_average=not training)
#         elif self.norm_type == "layer":
#             x = self.norm_in(x)
#         x = self.activation(x)
        
#         # Process through residual blocks
#         for block in self.blocks:
#             x = block(x, training=training)
        
#         # Output projection
#         x = self.conv_out(x)
#         if self.norm_type == "batch":
#             x = self.norm_out(x, use_running_average=not training)
#         elif self.norm_type == "layer":
#             x = self.norm_out(x)
        
#         return x

In [38]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
from typing import Callable, Tuple, Any

class ResBlock(nnx.Module):
    """A residual block with 3D convolutions that properly handles BatchNorm state."""
    
    def __init__(
        self,
        channels: int,
        kernel_size: Tuple[int, int, int],
        activation: Callable = jax.nn.relu,
        norm_type: str = "batch",
        use_residual: bool = True,
        rngs: nnx.Rngs = nnx.Rngs(0),
    ):
        super().__init__()
        self.use_residual = use_residual
        self.activation = activation
        
        # First convolution layer
        self.conv1 = nnx.Conv(
            in_features=channels,
            out_features=channels,
            kernel_size=kernel_size,
            strides=(1, 1, 1),
            padding="CIRCULAR",
            use_bias=False,
            kernel_init=nnx.initializers.variance_scaling(2.0, 'fan_in', 'normal'),
            rngs=rngs,
        )
        
        # Second convolution layer
        self.conv2 = nnx.Conv(
            in_features=channels,
            out_features=channels,
            kernel_size=kernel_size,
            strides=(1, 1, 1),
            padding="CIRCULAR",
            use_bias=False,
            kernel_init=nnx.initializers.variance_scaling(2.0, 'fan_in', 'normal'),
            rngs=rngs,
        )
        
        # Normalization layers
        if norm_type == "batch":
            self.norm1 = nnx.BatchNorm(
                num_features=channels,
                use_running_average=False,  # Will be set during call
                momentum=0.9,
                epsilon=1e-5,
                rngs=rngs,
            )
            self.norm2 = nnx.BatchNorm(
                num_features=channels,
                use_running_average=False,  # Will be set during call
                momentum=0.9,
                epsilon=1e-5,
                rngs=rngs,
            )
        elif norm_type == "layer":
            self.norm1 = nnx.LayerNorm(features=channels, epsilon=1e-5, rngs=rngs)
            self.norm2 = nnx.LayerNorm(features=channels, epsilon=1e-5, rngs=rngs)
        else:
            self.norm1 = lambda x, **kwargs: x  # Identity function
            self.norm2 = lambda x, **kwargs: x  # Identity function
    
    def __call__(self, x, training: bool = True):
        shortcut = x
        
        # First conv block
        x = self.conv1(x)
        if isinstance(self.norm1, nnx.BatchNorm):
            x = self.norm1(x, use_running_average=not training)
        else:
            x = self.norm1(x)
        x = self.activation(x)
        
        # Second conv block
        x = self.conv2(x)
        if isinstance(self.norm2, nnx.BatchNorm):
            x = self.norm2(x, use_running_average=not training)
        else:
            x = self.norm2(x)
        
        # Add residual connection if specified
        if self.use_residual:
            x = x + shortcut
            
        return self.activation(x)


class ResNet(nnx.Module):
    """3D ResNet implementation for voxel data with fixed BatchNorm handling."""
    
    def __init__(
        self,
        in_channels: int,
        hidden_channels: int,
        out_channels: int,
        n_hidden: int,
        kernel_size: Tuple[int, int, int] = (3, 3, 3),
        strides: int = 1,
        rngs: nnx.Rngs = nnx.Rngs(0),
        activation: Callable = jax.nn.relu,
        norm_type: str = "batch",
        use_residual: bool = True,
    ):
        super().__init__()
        
        # Input convolution
        self.input_conv = nnx.Conv(
            in_features=in_channels,
            out_features=hidden_channels,
            kernel_size=kernel_size,
            strides=(1, 1, 1),
            padding="CIRCULAR",
            use_bias=True,
            kernel_init=nnx.initializers.variance_scaling(1.0, 'fan_in', 'uniform'),
            bias_init=nnx.initializers.zeros,
            rngs=rngs,
        )
        
        # Input normalization
        if norm_type == "batch":
            self.input_norm = nnx.BatchNorm(
                num_features=hidden_channels,
                use_running_average=False,  # Will be set during call
                momentum=0.9,
                epsilon=1e-5,
                rngs=rngs,
            )
        elif norm_type == "layer":
            self.input_norm = nnx.LayerNorm(features=hidden_channels, epsilon=1e-5, rngs=rngs)
        else:
            self.input_norm = lambda x, **kwargs: x  # Identity function
            
        # Better activation function for numerical stability
        if activation == jax.nn.relu:
            self.activation = lambda x: jax.nn.leaky_relu(x, negative_slope=0.01)
        else:
            self.activation = activation
        
        # Create residual blocks
        self.res_blocks = []
        for _ in range(n_hidden):
            self.res_blocks.append(
                ResBlock(
                    channels=hidden_channels,
                    kernel_size=kernel_size,
                    activation=self.activation,
                    norm_type=norm_type,
                    use_residual=use_residual,
                    rngs=rngs,
                )
            )
        
        # Output convolution
        self.output_conv = nnx.Conv(
            in_features=hidden_channels,
            out_features=out_channels,
            kernel_size=kernel_size,
            strides=(1, 1, 1),
            padding="CIRCULAR",
            use_bias=True,
            kernel_init=nnx.initializers.variance_scaling(1.0, 'fan_out', 'uniform'),
            bias_init=nnx.initializers.zeros,
            rngs=rngs,
        )
        
        # Save properties
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.use_residual = use_residual
        self.norm_type = norm_type
        
        # Optional skip adapter for global residual connection
        if in_channels != out_channels and use_residual:
            self.skip_adapter = nnx.Conv(
                in_features=in_channels,
                out_features=out_channels,
                kernel_size=(1, 1, 1),
                strides=(1, 1, 1),
                padding="CIRCULAR",
                use_bias=False,
                kernel_init=nnx.initializers.variance_scaling(1.0, 'fan_in', 'uniform'),
                rngs=rngs,
            )
    
    def __call__(self, x, training: bool = True):
        # Save input for potential skip connection
        input_x = x
        
        # Input projection
        x = self.input_conv(x)
        
        # Apply normalization with proper handling of BatchNorm
        if isinstance(self.input_norm, nnx.BatchNorm):
            x = self.input_norm(x, use_running_average=not training)
        else:
            x = self.input_norm(x)
            
        x = self.activation(x)
        
        # Process through residual blocks
        for res_block in self.res_blocks:
            x = res_block(x, training=training)
        
        # Output projection
        x = self.output_conv(x)
        
        # Add global skip connection if using residuals
        if self.use_residual:
            if self.in_channels == self.out_channels:
                x = x + input_x
            elif hasattr(self, 'skip_adapter'):
                x = x + self.skip_adapter(input_x)
        
        return x

In [39]:
model = ResNet(
    in_channels=5, 
    out_channels=1,
    hidden_channels=64,
    n_hidden=2,
    kernel_size=(3, 3, 3),
    strides=1,
    rngs=nnx.Rngs(0),
    use_residual=False,
)
architecture = "cnn"

In [40]:
temp_in = np.random.random((3,64,64,64,5))
temp_out = model(temp_in)

In [41]:
temp_out.min()

Array(-25.207756, dtype=float32)

### MLP + CNN

In [42]:
# if latent_init is None:
#     latent_dim = 0
# else:
#     latent_dim = latent_init.shape[-1]

# mlp = MLP(
#     d_in=5 + latent_dim,
#     d_out=8, 
#     d_hidden=64, 
#     n_hidden=4, 
#     rngs=nnx.Rngs(0)
# )

# cnn = CNN(
#     d_in=4 + latent_dim, 
#     d_out=8,
#     d_hidden=16,
#     n_hidden=1,
#     kernel_size=(3, 3, 3),
#     strides=1,
#     rngs=nnx.Rngs(0)
# )

# model = HybridNet(
#     mlp,
#     cnn,
#     d_out=1 + latent_dim, 
#     rngs=nnx.Rngs(0)
# )

# architecture = "mlp+cnn"

### GNN

on the fly

In [43]:
# with_latent = False

# model = AttentionGNN(
#     d_node=5 + with_latent,
#     d_edge=1,
#     d_query=32,
#     n_hidden=4,
#     d_out=1 + with_latent,
#     rngs=nnx.Rngs(0),
# )

# architecture = "gnn"

# training

In [44]:
total_steps = 100
learning_rate = 1e-3
# learning_rate = 1e-4
# learning_rate = 1e-5
# learning_rate = optax.cosine_decay_schedule(
#     init_value=1e-4, 
#     decay_steps=total_steps, 
#     alpha=0.1
# )
# learning_rate = optax.warmup_cosine_decay_schedule(
#     init_value=1e-5,
#     peak_value=1e-4,
#     end_value=1e-5,
#     warmup_steps=total_steps//5,
#     decay_steps=total_steps - total_steps//5, 
# )
clip_norm = 1

optimizer = nnx.Optimizer(
    model,
    optax.chain(
        optax.clip_by_global_norm(clip_norm),
        optax.adam(learning_rate, eps=1e-5)
    )
)

losses = []
# loss_fn = lambda model: particle_loss_fn(model, architecture)
loss_fn = lambda model: field_loss_fn(model, architecture)

In [45]:
for i in (pbar := tqdm.tqdm(range(total_steps))):  
    loss = train_step(model, optimizer, loss_fn)

    losses.append(loss)
    pbar.set_description(f"Loss: {loss:.4f}")

fig, ax = plt.subplots()
ax.plot(losses)
ax.set(yscale="log")

  0%|          | 0/100 [00:00<?, ?it/s]

using voxel MSE
dark matter and gas
Using CNN forces
Using learned pressure force
dark matter and gas
Using CNN forces
Using learned pressure force


UnexpectedTracerError: Encountered an unexpected tracer. A function transformed by JAX had a side effect, allowing for a reference to an intermediate value with type float32[64] wrapped in a DynamicJaxprTracer to escape the scope of the transformation.
JAX transformations require that functions explicitly return their outputs, and disallow saving intermediate values to global state.
The function being traced when the value leaked was _fn at /cluster/home/athomsen/flatiron/lib/python3.11/site-packages/equinox/_eval_shape.py:31 traced for jit.
------------------------------
The leaked intermediate value was created on line /cluster/home/athomsen/flatiron/lib/python3.11/site-packages/flax/nnx/nn/normalization.py:361:8 (BatchNorm.__call__). 
------------------------------
When the value was created, the final 5 stack frames (most recent last) excluding JAX-internal frames were:
------------------------------
/cluster/home/athomsen/flatiron/repos/JaxPM/jaxpm/hpm.py:729:44 (get_hpm_network_o

### checkpointing

In [ ]:
# # see https://flax.readthedocs.io/en/latest/guides/checkpointing.html
# # checkpoint_file = os.path.join(os.getcwd(), "checkpoints/mlp_sim_v1_late_time.jx")
# # checkpoint_file = os.path.join(os.getcwd(), "checkpoints/mlp_sim_v2.jx")
# # checkpoint_file = os.path.join(os.getcwd(), "checkpoints/mlp_sim_illustris_v1.jx")
# checkpoint_file = os.path.join(os.getcwd(), "checkpoints/mlp_sim_v3_full_weighted.jx")
# checkpointer = ocp.StandardCheckpointer()
# print(os.getcwd())

In [ ]:
# _, params = nnx.split(model)
# checkpointer.save(checkpoint_file, params, force=True)

In [ ]:
# abstract_model = nnx.eval_shape(lambda: model)
# graphdef, abstract_params = nnx.split(abstract_model)

# params = checkpointer.restore(checkpoint_file, abstract_params)
# model = nnx.merge(graphdef, params)

# check simulation

In [ ]:
diagnostics.run_simulations(train_dict, mesh_per_dim, pressure_model=model, gas_architecture=architecture)

# validation

In [ ]:
stop

In [ ]:
vali_dict = camels.load_CV_snapshots(
    "CV_1",
    # "CV_0",
    mesh_per_dim,
    parts_per_dim,
    i_snapshots=i_snapshots,
    return_hydro=True,
)

In [ ]:
run_simulations(vali_dict)